
# Himalayan `RawData` Exploratory Data Analysis (EDA)

This notebook performs a concise, reproducible EDA on the repository's `RawData` directory:
- `RawData/expeditions.csv`
- `RawData/members.csv`
- `RawData/peaks.csv`

It focuses on schema inspection, missing data, duplicates, summary statistics, cardinality, distributions, outliers, and domain-specific quality checks useful for downstream ML modeling.


In [ ]:

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)

RAW_DIR = Path('RawData')
FILES = {
    'expeditions': RAW_DIR / 'expeditions.csv',
    'members': RAW_DIR / 'members.csv',
    'peaks': RAW_DIR / 'peaks.csv',
}

missing = [str(p) for p in FILES.values() if not p.exists()]
if missing:
    raise FileNotFoundError(f'Missing raw data file(s): {missing}')

raw_data = {
    name: pd.read_csv(path, na_values=['NA', ''])
    for name, path in FILES.items()
}

DATE_COLUMNS = {
    'expeditions': ['basecamp_date', 'highpoint_date', 'termination_date']
}

for name, cols in DATE_COLUMNS.items():
    for col in cols:
        raw_data[name][col] = pd.to_datetime(raw_data[name][col], errors='coerce')

print('Loaded tables:')
for name, df in raw_data.items():
    print(f"- {name}: shape={df.shape}")


## 1) Dataset inventory, shape, and schema

In [ ]:

schema_rows = []
for name, df in raw_data.items():
    schema_rows.append({
        'table': name,
        'rows': df.shape[0],
        'columns': df.shape[1],
        'numeric_columns': df.select_dtypes(include='number').shape[1],
        'categorical_columns': df.select_dtypes(include=['object', 'category', 'bool']).shape[1],
        'datetime_columns': df.select_dtypes(include='datetime').shape[1],
    })

schema_summary = pd.DataFrame(schema_rows).sort_values('table').reset_index(drop=True)
display(schema_summary)

for name, df in raw_data.items():
    print()
    print(f"{name.upper()} columns:")
    display(pd.DataFrame({'column': df.columns, 'dtype': df.dtypes.astype(str)}))



**Interpretation:**
- `members` is the largest table and likely drives most row-level modeling tasks.
- `expeditions` includes multiple date and operational fields.
- `peaks` is a compact reference table for mountain-level metadata.


## 2) Missing values and duplicate analysis

In [ ]:

quality_rows = []
for name, df in raw_data.items():
    total_cells = df.shape[0] * df.shape[1]
    missing_cells = int(df.isna().sum().sum())
    duplicate_rows = int(df.duplicated().sum())
    quality_rows.append({
        'table': name,
        'missing_cells': missing_cells,
        'missing_pct_cells': round((missing_cells / total_cells) * 100, 2),
        'duplicate_rows': duplicate_rows,
        'duplicate_pct_rows': round((duplicate_rows / max(df.shape[0], 1)) * 100, 2),
    })

quality_summary = pd.DataFrame(quality_rows).sort_values('table').reset_index(drop=True)
display(quality_summary)

for name, df in raw_data.items():
    mv = (
        df.isna().mean().mul(100)
        .sort_values(ascending=False)
        .rename('missing_pct')
        .to_frame()
    )
    mv = mv[mv['missing_pct'] > 0]
    print()
    print(f"{name.upper()} - columns with missing values")
    display(mv.head(15) if not mv.empty else pd.DataFrame({'missing_pct': []}))



**Interpretation:**
- Missingness is expected for event-specific fields (e.g., death/injury heights, optional dates) and should usually be treated as *structural missingness* rather than random noise.
- Exact duplicate rows are checked at table level; if non-zero, duplicates should be reviewed before modeling.


## 3) Data types, descriptive statistics, and cardinality

In [ ]:

for name, df in raw_data.items():
    print()
    print(f"=== {name.upper()} : NUMERIC SUMMARY ===")
    numeric = df.select_dtypes(include='number')
    display(numeric.describe().T if not numeric.empty else pd.DataFrame())

    print()
    print(f"=== {name.upper()} : CATEGORICAL SUMMARY (top) ===")
    categorical = df.select_dtypes(include=['object', 'category', 'bool'])
    if not categorical.empty:
        display(categorical.describe(include='all').T)
    else:
        display(pd.DataFrame())

    card = (
        pd.DataFrame({
            'column': df.columns,
            'dtype': df.dtypes.astype(str),
            'unique_values': [df[c].nunique(dropna=True) for c in df.columns],
            'missing_pct': [round(df[c].isna().mean() * 100, 2) for c in df.columns]
        })
        .sort_values(['unique_values', 'column'], ascending=[False, True])
        .reset_index(drop=True)
    )
    print()
    print(f"=== {name.upper()} : CARDINALITY ===")
    display(card)


## 4) Distribution visualizations (numeric and categorical)

In [ ]:

for name, df in raw_data.items():
    numeric_cols = [c for c in df.select_dtypes(include='number').columns if df[c].notna().sum() > 0]
    if numeric_cols:
        n = len(numeric_cols)
        ncols = 3
        nrows = int(np.ceil(n / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.8 * nrows))
        axes = np.atleast_1d(axes).ravel()

        for i, col in enumerate(numeric_cols):
            sns.histplot(df[col].dropna(), kde=True, ax=axes[i], bins=30)
            axes[i].set_title(f'{name}: {col}')

        for j in range(i + 1, len(axes)):
            axes[j].axis('off')

        plt.tight_layout()
        plt.show()

for name, df in raw_data.items():
    cat_cols = [
        c for c in df.select_dtypes(include=['object', 'category', 'bool']).columns
        if 2 <= df[c].nunique(dropna=True) <= 20
    ]

    for col in cat_cols[:6]:
        vc = df[col].astype('string').fillna('Missing').value_counts().head(12)
        plt.figure(figsize=(8, 3.8))
        sns.barplot(x=vc.values, y=vc.index)
        plt.title(f'{name}: top categories in {col}')
        plt.xlabel('count')
        plt.ylabel(col)
        plt.tight_layout()
        plt.show()


## 5) Outlier review and notable data-quality issues

In [ ]:

def iqr_outlier_summary(df, table_name):
    rows = []
    for col in df.select_dtypes(include='number').columns:
        s = df[col].dropna()
        if s.shape[0] < 8 or s.nunique() < 4:
            continue
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr == 0:
            continue
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outliers = ((s < lower) | (s > upper)).sum()
        rows.append({
            'table': table_name,
            'column': col,
            'q1': round(float(q1), 3),
            'q3': round(float(q3), 3),
            'iqr': round(float(iqr), 3),
            'lower_bound': round(float(lower), 3),
            'upper_bound': round(float(upper), 3),
            'outlier_rows': int(outliers),
            'outlier_pct': round(100 * outliers / len(s), 2),
        })
    return pd.DataFrame(rows)

outlier_tables = []
for name, df in raw_data.items():
    outlier_tables.append(iqr_outlier_summary(df, name))

outlier_summary = pd.concat(outlier_tables, ignore_index=True)
outlier_summary = outlier_summary.sort_values(['outlier_pct', 'outlier_rows'], ascending=False)
display(outlier_summary)

for name, df in raw_data.items():
    numeric_cols = [c for c in df.select_dtypes(include='number').columns if df[c].notna().sum() > 0]
    if not numeric_cols:
        continue

    cols_to_plot = numeric_cols[:6]
    n = len(cols_to_plot)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    axes = np.atleast_1d(axes)
    for i, col in enumerate(cols_to_plot):
        sns.boxplot(y=df[col], ax=axes[i])
        axes[i].set_title(f'{name}: {col}')
    plt.tight_layout()
    plt.show()



**Outlier assumptions used:**
- Numeric outliers are flagged with the standard IQR rule (values below `Q1 - 1.5*IQR` or above `Q3 + 1.5*IQR`).
- In mountaineering data, many "outliers" can be valid rare events (e.g., very large teams, unusual heights, death/injury-related records) and should not be removed blindly.
- Prefer capping, robust scaling, or model choices tolerant to skew/heavy tails, depending on task.


## 6) Targeted domain analysis (dates, labels, and key fields)

In [ ]:

# Expeditions date quality and seasonal/year trends
exp = raw_data['expeditions'].copy()
exp['days_basecamp_to_highpoint'] = (exp['highpoint_date'] - exp['basecamp_date']).dt.days
exp['days_highpoint_to_termination'] = (exp['termination_date'] - exp['highpoint_date']).dt.days

print('Expeditions date interval summary:')
display(exp[['days_basecamp_to_highpoint', 'days_highpoint_to_termination']].describe().T)

plt.figure(figsize=(8, 3.8))
exp.groupby('year').size().plot()
plt.title('Expeditions per year')
plt.xlabel('year')
plt.ylabel('count')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 3.8))
sns.countplot(data=exp, x='season', order=exp['season'].value_counts().index)
plt.title('Season distribution in expeditions')
plt.tight_layout()
plt.show()

# Members outcome labels and demographics
mem = raw_data['members'].copy()
for col in ['died', 'success', 'oxygen_used', 'hired', 'injured']:
    print(f"{col} distribution:")
    display(mem[col].value_counts(dropna=False).rename_axis(col).to_frame('count'))

plt.figure(figsize=(8, 3.8))
sns.histplot(mem['age'].dropna(), bins=35, kde=True)
plt.title('Member age distribution')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 3.8))
(
    mem['citizenship']
    .fillna('Missing')
    .value_counts()
    .head(12)
    .sort_values()
    .plot(kind='barh')
)
plt.title('Top member citizenships')
plt.xlabel('count')
plt.tight_layout()
plt.show()

# Peaks reference data
pk = raw_data['peaks'].copy()
plt.figure(figsize=(8, 3.8))
sns.histplot(pk['height_metres'].dropna(), bins=30, kde=True)
plt.title('Peak height distribution (metres)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 3.8))
sns.countplot(data=pk, x='climbing_status', order=pk['climbing_status'].value_counts().index)
plt.title('Peak climbing status distribution')
plt.tight_layout()
plt.show()



## 7) Final summary and preprocessing/modeling recommendations

### Key findings
- The `RawData` directory contains three linked tables with different granularity (peak-level, expedition-level, member-level).
- Missingness is substantial in several event-specific fields and should be handled with column semantics in mind.
- Some numeric variables show skew and IQR outliers; many are likely domain-valid extremes.
- Boolean outcome/condition fields in `members` (e.g., `died`, `success`, `injured`) can support classification tasks.
- Date fields in `expeditions` enable duration-based features and temporal trend analysis.

### Recommended preprocessing for ML assignment
1. **Join strategy:** model at a chosen level (member or expedition) and join keys (`expedition_id`, `peak_id`) carefully.
2. **Missing values:** use semantic imputation (e.g., keep NA for not-applicable event fields, median for continuous variables when needed).
3. **Categoricals:** one-hot encode low/medium-cardinality fields; group rare categories for very high-cardinality fields.
4. **Numerics:** consider log/robust transforms for skewed counts/heights and robust models for outlier tolerance.
5. **Dates:** engineer duration, month/season, and historical trend features from parsed date columns.
6. **Class imbalance:** inspect label balance (especially `died`) and apply appropriate metrics/resampling if needed.

> This notebook does not modify the raw source files and uses repository-relative paths so it can be rerun from a clean kernel.
